In [ ]:
import json
import requests
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib

with open("licenses.json", "r", encoding="utf-8") as f:
    data = json.load(f)

licenses = data["licenses"]

texts = []
labels = []

print("🔍 Downloading SPDX license texts...")

for lic in licenses:
    url = lic["detailsUrl"]

    try:
        res = requests.get(url)
        full = res.json()

        if "licenseText" in full:
            text = full["licenseText"]
            label = full["licenseId"]

            texts.append(text)
            labels.append(label)

            print("✔ Loaded:", label)
        else:
            print("❌ No licenseText:", lic["licenseId"])

    except Exception as e:
        print("⚠ Error loading:", lic["licenseId"], str(e))


df = pd.DataFrame({"text": texts, "label": labels})

print("\n📌 Total licenses downloaded:", len(df))


X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", max_features=50000)),
    ("svc", LinearSVC())
])

print("\n🚀 Training model...")
model.fit(X_train, y_train)

preds = model.predict(X_test)
print("\n📊 Classification Report")
print(classification_report(y_test, preds))


joblib.dump(model, "license_classifier_model.joblib")

print("\n💾 Model saved as license_classifier_model.joblib")


🔍 Downloading SPDX license texts...
✔ Loaded: 0BSD
✔ Loaded: 3D-Slicer-1.0
✔ Loaded: AAL
✔ Loaded: Abstyles
✔ Loaded: AdaCore-doc
✔ Loaded: Adobe-2006
✔ Loaded: Adobe-Display-PostScript
✔ Loaded: Adobe-Glyph
✔ Loaded: Adobe-Utopia
✔ Loaded: ADSL
⚠ Error loading: Advanced-Cryptics-Dictionary Expecting value: line 1 column 1 (char 0)
✔ Loaded: AFL-1.1
✔ Loaded: AFL-1.2
✔ Loaded: AFL-2.0
✔ Loaded: AFL-2.1
✔ Loaded: AFL-3.0
✔ Loaded: Afmparse
✔ Loaded: AGPL-1.0
✔ Loaded: AGPL-1.0-only
✔ Loaded: AGPL-1.0-or-later
✔ Loaded: AGPL-3.0
✔ Loaded: AGPL-3.0-only
✔ Loaded: AGPL-3.0-or-later
✔ Loaded: Aladdin
✔ Loaded: AMD-newlib
✔ Loaded: AMDPLPA
✔ Loaded: AML
✔ Loaded: AML-glslang
✔ Loaded: AMPAS
✔ Loaded: ANTLR-PD
✔ Loaded: ANTLR-PD-fallback
✔ Loaded: any-OSI
✔ Loaded: any-OSI-perl-modules
✔ Loaded: Apache-1.0
✔ Loaded: Apache-1.1
✔ Loaded: Apache-2.0
✔ Loaded: APAFML
✔ Loaded: APL-1.0
✔ Loaded: App-s2p
✔ Loaded: APSL-1.0
✔ Loaded: APSL-1.1
✔ Loaded: APSL-1.2
✔ Loaded: APSL-2.0
✔ Loaded: Arphic-1

C:\Users\adel mohamedll\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(y_pred, input_name="y_pred")
C:\Users\adel mohamedll\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  ys_types = set(type_of_target(x) for x in ys)
C:\Users\adel mohamedll\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(y_pred, input_name="y_pred")
C:\Users\adel mohamedll\AppData\R

In [ ]:
import pandas as pd
import random
import nltk
from nltk.corpus import wordnet

nltk.download('wordnet')
nltk.download('omw-1.4')

def synonym_replacement(text, n=2):
    words = text.split()
    new_words = words.copy()
    random_word_list = list(set(words))
    random.shuffle(random_word_list)
    num_replaced = 0
    
    for random_word in random_word_list:
        synonyms = wordnet.synsets(random_word)
        if synonyms:
            synonym = synonyms[0].lemmas()[0].name()
            if synonym != random_word:
                new_words = [synonym if w==random_word else w for w in new_words]
                num_replaced += 1
        if num_replaced >= n:
            break
    return ' '.join(new_words)

def random_deletion(text, p=0.1):
    words = text.split()
    if len(words) == 1:
        return text
    new_words = [w for w in words if random.uniform(0,1) > p]
    if len(new_words) == 0:
        return random.choice(words)
    return ' '.join(new_words)

def random_insertion(text, n=1):
    words = text.split()
    new_words = words.copy()
    for _ in range(n):
        word_to_insert = random.choice(words)
        synonyms = wordnet.synsets(word_to_insert)
        if synonyms:
            synonym = synonyms[0].lemmas()[0].name()
            insert_pos = random.randint(0, len(new_words))
            new_words.insert(insert_pos, synonym)
    return ' '.join(new_words)

def augment_text(text):
    augmented_texts = []
    augmented_texts.append(synonym_replacement(text, n=2))
    augmented_texts.append(random_deletion(text, p=0.1))
    augmented_texts.append(random_insertion(text, n=2))
    return list(set(augmented_texts))  # إزالة التكرار


def augment_csv(input_csv, output_csv, text_column='license_text'):
    df = pd.read_csv(input_csv)
    augmented_rows = []

    for _, row in df.iterrows():
        original_text = row[text_column]
        augmented_rows.append({'license_text': original_text, 'type': 'original'})
        augmented_versions = augment_text(original_text)
        for aug_text in augmented_versions:
            augmented_rows.append({'license_text': aug_text, 'type': 'augmented'})

    df_augmented = pd.DataFrame(augmented_rows)
    df_augmented.to_csv(output_csv, index=False)
    print(f"Augmented CSV saved to {output_csv}")


if __name__ == "__main__":
    augment_csv("all_licenses.csv", "licenses_augmented_offline.csv", text_column='name')


[nltk_data] Downloading package wordnet to C:\Users\adel
[nltk_data]     mohamedll\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\adel
[nltk_data]     mohamedll\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Augmented CSV saved to licenses_augmented_offline.csv


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import joblib

df = pd.read_csv("licenses_augmented_offline.csv")

df = df.rename(columns={'type': 'label'})  

X_train, X_test, y_train, y_test = train_test_split(
    df['license_text'], df['label'], test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


clf = LogisticRegression(max_iter=500)
clf.fit(X_train_tfidf, y_train)

y_pred = clf.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


joblib.dump(clf, "license_classifier.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("Model and vectorizer saved successfully!")


Accuracy: 0.7279279279279279
              precision    recall  f1-score   support

   augmented       0.73      0.99      0.84       407
    original       0.00      0.00      0.00       148

    accuracy                           0.73       555
   macro avg       0.37      0.50      0.42       555
weighted avg       0.54      0.73      0.62       555

Model and vectorizer saved successfully!


In [ ]:
import joblib

clf = joblib.load("license_classifier.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")

new_text = ["This software is licensed under MIT License."]

new_tfidf = vectorizer.transform(new_text)

pred = clf.predict(new_tfidf)
print("Predicted label:", pred[0])


Predicted label: augmented
